In [ ]:
# 1. Import libraries
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import seaborn as sns
from rasterstats import zonal_stats


In [ ]:
# 2. Set fire inputs and paths
VIIRS_GPKG = Path("viirs_global_375m.gpkg")
CBI_DIR = Path(".")
OUTPUT_DIR = Path("outputs")

FIRE_NAME = "Riverside"
FIRE_ID = 38454
FIRE_DATE = "2020-01-01"  # YYYY-MM-DD

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 3. Helper functions
def duration_category(duration_days):
    """Classify fires into short vs long duration groups."""
    if pd.isna(duration_days):
        return "unknown"
    return "short" if duration_days < 7 else "long"


def get_cbi_path(cbi_dir, fire_date):
    year = pd.to_datetime(fire_date).year
    cbi_path = Path(cbi_dir) / f"{year}_pnw_firedv2_bccbi_epsg5070.tif"
    if not cbi_path.exists():
        raise FileNotFoundError(f"No CBI raster for year {year}: {cbi_path}")
    return cbi_path


In [ ]:
# 4. Load VIIRS and isolate one fire
viirs = gpd.read_file(VIIRS_GPKG)
viirs_singlefire = viirs.loc[viirs["id"] == FIRE_ID].copy()

if viirs_singlefire.empty:
    raise ValueError(f"No VIIRS records found for id={FIRE_ID}")

summary = pd.DataFrame([
    {
        "fire_name": FIRE_NAME,
        "fire_id": FIRE_ID,
        "fire_date": FIRE_DATE,
        "n_cells": len(viirs_singlefire),
        "frp_norm_csum_mean": viirs_singlefire["frp_norm_csum"].mean(),
        "frp_norm_csum_median": viirs_singlefire["frp_norm_csum"].median(),
        "frp_norm_csum_min": viirs_singlefire["frp_norm_csum"].min(),
        "frp_norm_csum_max": viirs_singlefire["frp_norm_csum"].max(),
        "obs_duration_mean": viirs_singlefire["obs_duration"].mean(),
        "obs_duration_median": viirs_singlefire["obs_duration"].median(),
        "obs_duration_min": viirs_singlefire["obs_duration"].min(),
        "obs_duration_max": viirs_singlefire["obs_duration"].max(),
    }
])

summary


In [ ]:
# 5. Join per-cell CBI stats using zonal statistics
cbi_path = get_cbi_path(CBI_DIR, FIRE_DATE)
with rasterio.open(cbi_path) as src:
    cbi_array = src.read(1, masked=True).filled(np.nan)
    cbi_transform = src.transform

downsample_stats = zonal_stats(
    viirs_singlefire,
    cbi_array,
    affine=cbi_transform,
    stats=["mean", "median", "min", "max", "percentile_95", "std"],
    nodata=np.nan,
    all_touched=True,
)

viirs_singlefire["cbi_mean"] = [s["mean"] for s in downsample_stats]
viirs_singlefire["cbi_stdev"] = [s["std"] for s in downsample_stats]
viirs_singlefire["cbi_median"] = [s["median"] for s in downsample_stats]
viirs_singlefire["cbi_min"] = [s["min"] for s in downsample_stats]
viirs_singlefire["cbi_max"] = [s["max"] for s in downsample_stats]
viirs_singlefire["cbi_95"] = [s["percentile_95"] for s in downsample_stats]

viirs_singlefire["log_cfrp"] = np.log(viirs_singlefire["frp_norm_csum"] + 1)
viirs_singlefire["duration_group"] = viirs_singlefire["obs_duration"].apply(duration_category)

bins = [0, 500, 1000, 2000, viirs_singlefire["frp_norm_csum"].max()]
labels = ["<500", "500-1000", "1000-2000", "2000+"]
viirs_singlefire["intensity_class"] = pd.cut(
    viirs_singlefire["frp_norm_csum"],
    bins=bins,
    labels=labels,
    include_lowest=True,
)

viirs_singlefire.head()


In [ ]:
# 6. Save outputs
safe_fire_name = FIRE_NAME.replace(" ", "_")
gpkg_path = OUTPUT_DIR / f"{safe_fire_name}_viirs_singlefire_metrics.gpkg"
summary_path = OUTPUT_DIR / f"{safe_fire_name}_summary_stats.csv"

viirs_singlefire.to_file(gpkg_path, driver="GPKG")
summary.to_csv(summary_path, index=False)

print("✅ Wrote outputs:")
print(f"  - {gpkg_path}")
print(f"  - {summary_path}")


In [ ]:
# 7. Create figures
fig, axes = plt.subplots(1, 3, figsize=(16, 6), constrained_layout=True)
viirs_singlefire.plot(column="log_cfrp", ax=axes[0], cmap="viridis", legend=True, linewidth=0)
axes[0].set_title("log_cfrp")
axes[0].set_axis_off()

viirs_singlefire.plot(column="cbi_mean", ax=axes[1], cmap="magma", legend=True, linewidth=0, vmin=0, vmax=3)
axes[1].set_title("cbi_mean")
axes[1].set_axis_off()

viirs_singlefire.plot(column="obs_duration", ax=axes[2], cmap="plasma", legend=True, linewidth=0)
axes[2].set_title("obs_duration")
axes[2].set_axis_off()

maps_path = OUTPUT_DIR / f"{safe_fire_name}_maps_logcfrp_cbi_duration.png"
fig.savefig(maps_path, dpi=200)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(data=viirs_singlefire, x="log_cfrp", y="cbi_mean", hue="duration_group", alpha=0.6, ax=ax)
ax.set_title("log_cfrp vs cbi_mean by duration")
scatter_path = OUTPUT_DIR / f"{safe_fire_name}_scatter_logcfrp_vs_cbimean.png"
fig.savefig(scatter_path, dpi=200)
plt.close(fig)

gdf_clean = viirs_singlefire.dropna(subset=["cbi_mean", "frp_norm_csum", "duration_group", "intensity_class"])
fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(
    data=gdf_clean,
    x="intensity_class",
    y="cbi_mean",
    hue="duration_group",
    palette="viridis",
    ax=ax,
)
ax.set_ylim(0, 3)
ax.set_xlabel("cFRP intensity class")
ax.set_ylabel("cbi_mean")
ax.set_title("CBI by cFRP class grouped by duration")
boxplot_path = OUTPUT_DIR / f"{safe_fire_name}_boxplot_cfrpclass_cbimean_duration.png"
fig.savefig(boxplot_path, dpi=200)
plt.close(fig)

print("✅ Wrote figures:")
print(f"  - {maps_path}")
print(f"  - {scatter_path}")
print(f"  - {boxplot_path}")
